# 🐺 WolfPicture Printify Studio

AI görsellerini adım adım **Printify baskısına hazırlar**.

1. Kurulum  
2. Ayarlar  
3. Görsel yükleme  
4. Ham analiz  
5. Tişört rengi önerisi  
6. Baskı boyutu önerisi  
7. Arka plan silme  
8. Real-ESRGAN upscale  
9. Son kalite kontrolü  
10. Mockup  
11. Toplu rapor  
12. ZIP indirme

> Kod hücreleri gizlidir. Sırayla ▶️ düğmesine bas.

In [ ]:
#@title 1️⃣ Kurulum
import os,sys,subprocess,pkgutil,json,zipfile
from pathlib import Path
pkgs={'cv2':'opencv-python-headless','PIL':'Pillow','numpy':'numpy','rembg':'rembg','onnxruntime':'onnxruntime','pandas':'pandas'}
for mod,pkg in pkgs.items():
    if pkgutil.find_loader(mod) is None: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('✅ Kurulum tamamlandı.')

## 2️⃣ Ayarlar
Varsayılan değerler güvenlidir.

In [ ]:
#@title 2️⃣ Ayarlar
target_dpi=300 #@param {type:"integer"}
recommended_print_width_cm=30 #@param {type:"integer"}
max_file_size_mb=100 #@param {type:"integer"}
remove_background=True #@param {type:"boolean"}
upscale_factor=4 #@param [2,4]
create_mockups=True #@param {type:"boolean"}
use_face_restore=False #@param {type:"boolean"}
ROOT=Path('/content/wolfpicture'); RAW=ROOT/'01_raw'; REM=ROOT/'02_removed'; UPS=ROOT/'03_upscaled'; MOCK=ROOT/'04_mockups'; REP=ROOT/'05_reports'
for d in [RAW,REM,UPS,MOCK,REP]: d.mkdir(parents=True,exist_ok=True)
print('✅ Ayarlar hazır. Face Restore kapalı.')

## 3️⃣ Görselleri Yükle
PNG, JPG, JPEG veya WEBP seç.

In [ ]:
#@title 3️⃣ Görselleri Yükle
from google.colab import files
uploaded=files.upload(); allowed={'.png','.jpg','.jpeg','.webp'}
for name,data in uploaded.items():
    if Path(name).suffix.lower() in allowed: (RAW/Path(name).name).write_bytes(data); print('✅',name)
raw_files=sorted([p for p in RAW.iterdir() if p.suffix.lower() in allowed])
print('📦 Görsel sayısı:',len(raw_files))

## 4️⃣ Analiz Motoru
Keskinlik, koyuluk, detay, şeffaflık ve baskı ölçüsü hesaplanır.

In [ ]:
#@title 4️⃣ Analiz Motoru
import cv2,numpy as np,pandas as pd
from PIL import Image,ImageDraw
from IPython.display import display

def rgba(path):
    with Image.open(path) as im:return np.array(im.convert('RGBA'))
def small(a,limit=1600):
    h,w=a.shape[:2]; s=min(1,limit/max(h,w)); return a if s==1 else cv2.resize(a,(int(w*s),int(h*s)),interpolation=cv2.INTER_AREA)
def parts(a):return a[:,:,:3],a[:,:,3],a[:,:,3]>20
def sharp(a):
    rgb,al,m=parts(a)
    if not np.any(m):return 0
    g=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY); return float(np.var(cv2.Laplacian(g,cv2.CV_64F)[m]))
def metrics(path):
    full=rgba(path); a=small(full); rgb,al,m=parts(a); h,w=full.shape[:2]
    g=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY); vg=g[m] if np.any(m) else np.array([0])
    dark=float((vg<45).mean()*100); edge=cv2.Canny(g,80,180); edge[~m]=0; detail=float((edge>0).sum()/max(1,m.sum())*100)
    bw=max(2,int(min(al.shape)*.01)); border=np.concatenate([al[:bw,:].ravel(),al[-bw:,:].ravel(),al[:,:bw].ravel(),al[:,-bw:].ravel()]); border=float((border>10).mean()*100)
    sh=sharp(a); mb=os.path.getsize(path)/(1024**2); mx=w/target_dpi*2.54
    mn=28 if detail>=24 else 24 if detail>=18 else 21 if detail>=10 else 18
    if dark>=40:mn+=2
    rec=min(mx,max(mn+2,recommended_print_width_cm)); fail=0;warn=0
    if sh<35:fail+=1
    elif sh<70:warn+=1
    if dark>35:warn+=1
    if detail>24:warn+=1
    if min(w,h)<2000:fail+=1
    elif min(w,h)<3000:warn+=1
    if mb>max_file_size_mb:fail+=1
    if fail>=2:code,title,score='FAIL','❌ YENİDEN İŞLE',45
    elif fail==1 or warn>=3:code,title,score='CHECK','⚠️ KONTROL ET',72
    elif warn:code,title,score='READY_CHECK','✅ BASKIYA UYGUN — KÜÇÜK KONTROL',86 if warn>=2 else 94
    else:code,title,score='READY','✅ BASKIYA HAZIR',100
    return {'file':str(path),'width':w,'height':h,'sharpness':sh,'dark':dark,'detail':detail,'border':border,'file_mb':mb,'min_cm':round(mn,1),'recommended_cm':round(rec,1),'max_cm':round(mx,1),'code':code,'decision':title,'score':score}
def contrast(a,bg):
    rgb,al,m=parts(a)
    if not np.any(m):return 0
    af=al.astype(np.float32)/255; back=np.zeros_like(rgb,dtype=np.float32); back[:]=bg
    comp=rgb.astype(np.float32)*af[...,None]+back*(1-af[...,None]); gd=cv2.cvtColor(comp.astype(np.uint8),cv2.COLOR_RGB2GRAY); bggray=int(.299*bg[0]+.587*bg[1]+.114*bg[2])
    return round(max(0,min(100,float(np.mean(np.abs(gd.astype(float)-bggray)[m]))/1.15)),1)
print('✅ Analiz motoru hazır.')

## 5️⃣ Ham Görsel Analizi

In [ ]:
#@title 5️⃣ Ham Analiz
raw_results=[]
for p in raw_files:
    m=metrics(p);raw_results.append(m);print();print('='*70);print('🖼️',p.name);print('🎯',m['decision'],m['score'],'/100');print(f"📐 {m['width']}×{m['height']} | 🌑 %{m['dark']:.1f} | 🧵 %{m['detail']:.1f}");print(f"📏 Minimum {m['min_cm']} cm | Önerilen {m['recommended_cm']} cm")

## 6️⃣ Tişört Rengi Önerisi

In [ ]:
#@title 6️⃣ Tişört Renklerini Puanla
COLORS={'Beyaz':(255,255,255),'Krem':(242,232,210),'Açık Gri':(205,205,205),'Kum':(210,190,150),'Siyah':(20,20,20),'Koyu Gri':(65,65,65),'Lacivert':(25,35,70),'Bordo':(100,25,40),'Orman Yeşili':(45,70,45),'Kahverengi':(85,55,35)}
color_results={}
for p in raw_files:
    a=small(rgba(p)); ranked=sorted([(n,contrast(a,c)) for n,c in COLORS.items()],key=lambda x:x[1],reverse=True);color_results[p.name]=ranked
    print();print('='*70);print('👕',p.name);print('🏆 En iyi:',', '.join(f'{n} ({s})' for n,s in ranked[:4]));print('🚫 Zayıf:',', '.join(f'{n} ({s})' for n,s in ranked[-3:]))

## 7️⃣ Baskı Boyutu Önerisi

In [ ]:
#@title 7️⃣ Baskı Boyutu
for m in raw_results:
    print();print('='*70);print('📄',Path(m['file']).name);print('📏 Minimum:',m['min_cm'],'cm');print('✨ Önerilen:',m['recommended_cm'],'cm');print('🧮 300 DPI maksimum:',m['max_cm'],'cm');print('⚠️ Küçük göğüs baskısı önerilmez.' if m['detail']>24 else '✅ Orta veya büyük baskıya uygundur.')

## 8️⃣ Arka Planı Sil

In [ ]:
#@title 8️⃣ Arka Planı Sil
from rembg import remove
removed=[]
for p in raw_files:
    out=REM/f'{p.stem}_rembg.png'
    if remove_background:out.write_bytes(remove(p.read_bytes()))
    else:Image.open(p).convert('RGBA').save(out)
    removed.append(out);print('✅',out.name)

## 9️⃣ Real-ESRGAN Upscale

In [ ]:
#@title 9️⃣ Real-ESRGAN Kurulumu
import urllib.request
repo=Path('/content/Real-ESRGAN')
if not repo.exists():subprocess.run(['git','clone','-q','https://github.com/xinntao/Real-ESRGAN.git',str(repo)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','basicsr','facexlib','gfpgan'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements.txt')],check=True)
subprocess.run([sys.executable,'setup.py','develop','-q'],cwd=repo,check=True)
(repo/'weights').mkdir(exist_ok=True);wf=repo/'weights'/'RealESRGAN_x4plus.pth'
if not wf.exists():urllib.request.urlretrieve('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth',wf)
print('✅ Real-ESRGAN hazır.')

In [ ]:
#@title 9️⃣ Upscale İşlemi
upscaled=[]
for p in removed:
    subprocess.run([sys.executable,str(repo/'inference_realesrgan.py'),'-n','RealESRGAN_x4plus','-i',str(p),'-o',str(UPS),'-s',str(upscale_factor),'--ext','png'],cwd=repo,check=True)
    gen=UPS/f'{p.stem}_out.png';final=UPS/f'{p.stem}_upscaled.png'
    if gen.exists():gen.rename(final)
    else:
        cand=list(UPS.glob(f'{p.stem}*'))
        if not cand:raise FileNotFoundError(p.name)
        cand[0].rename(final)
    with Image.open(final) as im:im.convert('RGBA').save(final,dpi=(target_dpi,target_dpi))
    upscaled.append(final);print('✅',final.name)

## 🔟 Son Kalite Kontrolü

In [ ]:
#@title 🔟 Son Kontrol
final_results=[]
for p in upscaled:
    m=metrics(p);final_results.append(m);print();print('='*70);print('🐺',p.name);print('🎯',m['decision'],m['score'],'/100');print(f"📐 {m['width']}×{m['height']} | 📦 {m['file_mb']:.2f} MB");print(f"📏 Minimum {m['min_cm']} | Önerilen {m['recommended_cm']} | Maksimum {m['max_cm']} cm")

## 1️⃣1️⃣ Mockup Oluştur

In [ ]:
#@title 1️⃣1️⃣ Mockup
def mockup(design,bg,name,out,canvas=(1400,1600)):
    d=Image.open(design).convert('RGBA');base=Image.new('RGB',canvas,bg);d.thumbnail((int(canvas[0]*.62),int(canvas[1]*.58)),Image.Resampling.LANCZOS);x=(canvas[0]-d.width)//2;y=int(canvas[1]*.22);base.paste(d,(x,y),d);dr=ImageDraw.Draw(base);dr.rounded_rectangle([60,50,canvas[0]-60,canvas[1]-50],radius=25,outline=(130,130,130),width=4);dr.text((80,canvas[1]-90),name+' tişört',fill=(90,90,90));base.save(out,quality=95)
mockups=[]
if create_mockups:
    for p in upscaled:
        raw=next((k for k in color_results if Path(k).stem in p.name),None);best=[n for n,_ in color_results.get(raw,[])[:4]] or ['Beyaz','Krem','Açık Gri','Siyah']
        for n in best:
            out=MOCK/f"{p.stem}_{n.replace(' ','_')}.jpg";mockup(p,COLORS[n],n,out);mockups.append(out)
    print('✅ Mockup sayısı:',len(mockups))
    for f in mockups[:8]:display(Image.open(f).resize((280,320)))

## 1️⃣2️⃣ Toplu Rapor

In [ ]:
#@title 1️⃣2️⃣ Rapor
rows=[]
for m in final_results:
    name=Path(m['file']).name;raw=next((k for k in color_results if Path(k).stem in name),None);best=', '.join(n for n,_ in color_results.get(raw,[])[:4]);rows.append({'Dosya':name,'Karar':m['decision'],'Puan':m['score'],'Boyut':f"{m['width']}×{m['height']}",'Minimum cm':m['min_cm'],'Önerilen cm':m['recommended_cm'],'En iyi renkler':best,'MB':round(m['file_mb'],2)})
df=pd.DataFrame(rows);display(df);(REP/'report.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf-8');df.to_csv(REP/'report.csv',index=False,encoding='utf-8-sig');print('✅ Rapor hazır.')

## 1️⃣3️⃣ Sonuçları İndir

In [ ]:
#@title 1️⃣3️⃣ ZIP İndir
from google.colab import files
z=Path('/content/WolfPicture_Printify_Ready.zip')
if z.exists():z.unlink()
with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as out:
    for folder in [UPS,MOCK,REP]:
        for f in folder.rglob('*'):
            if f.is_file():out.write(f,f.relative_to(ROOT))
print('✅ ZIP hazır.');files.download(str(z))

# ✅ Tamamlandı
Şeffaf PNG, upscale dosyaları, renk ve boyut önerileri, mockuplar ve raporlar hazırdır. Nihai yerleşimi Printify Product Creator içinde kontrol et.